In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
from tensorflow.keras import  layers , models
from sklearn.metrics import classification_report , confusion_matrix
import seaborn as sns

In [25]:
train_df = pd.read_csv('sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test.csv')

In [26]:
train_df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,3,107,118,127,134,139,143,146,150,153,...,207,207,207,207,206,206,206,204,203,202
1,6,155,157,156,156,156,157,156,158,158,...,69,149,128,87,94,163,175,103,135,149
2,2,187,188,188,187,187,186,187,188,187,...,202,201,200,199,198,199,198,195,194,195
3,2,211,211,212,212,211,210,211,210,210,...,235,234,233,231,230,226,225,222,229,163
4,13,164,167,170,172,176,179,180,184,185,...,92,105,105,108,133,163,157,163,164,179


In [27]:
y_train = train_df['label'].values

In [28]:
test_df.head()


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,6,149,149,150,150,150,151,151,150,151,...,138,148,127,89,82,96,106,112,120,107
1,5,126,128,131,132,133,134,135,135,136,...,47,104,194,183,186,184,184,184,182,180
2,10,85,88,92,96,105,123,135,143,147,...,68,166,242,227,230,227,226,225,224,222
3,0,203,205,207,206,207,209,210,209,210,...,154,248,247,248,253,236,230,240,253,255
4,3,188,191,193,195,199,201,202,203,203,...,26,40,64,48,29,46,49,46,46,53


In [29]:
y_test = test_df['label'].values
y_test

array([ 6,  5, 10, ...,  2,  4,  2], shape=(7172,))

In [30]:
X_train = train_df.drop('label',axis=1).values
X_test = test_df.drop('label',axis=1).values

In [31]:
X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

In [32]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

In [33]:
[y_train,y_test]

[array([ 3,  6,  2, ..., 18, 17, 23], shape=(27455,)),
 array([ 6,  5, 10, ...,  2,  4,  2], shape=(7172,))]

In [34]:
np.concatenate([y_train,y_test])

array([3, 6, 2, ..., 2, 4, 2], shape=(34627,))

In [35]:
np.unique(np.concatenate([y_train,y_test]))

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24])

In [36]:
unique_labels = sorted(np.unique(np.concatenate([y_train,y_test])))
unique_labels

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24)]

In [37]:
label_map = {old: new for new, old in enumerate(unique_labels)}
label_map

{np.int64(0): 0,
 np.int64(1): 1,
 np.int64(2): 2,
 np.int64(3): 3,
 np.int64(4): 4,
 np.int64(5): 5,
 np.int64(6): 6,
 np.int64(7): 7,
 np.int64(8): 8,
 np.int64(10): 9,
 np.int64(11): 10,
 np.int64(12): 11,
 np.int64(13): 12,
 np.int64(14): 13,
 np.int64(15): 14,
 np.int64(16): 15,
 np.int64(17): 16,
 np.int64(18): 17,
 np.int64(19): 18,
 np.int64(20): 19,
 np.int64(21): 20,
 np.int64(22): 21,
 np.int64(23): 22,
 np.int64(24): 23}

In [38]:
y_train = np.array([label_map[y] for y in y_train])
y_test = np.array([label_map[y] for y in y_test])

In [39]:
len(unique_labels)

24

In [45]:
num_classes = len(unique_labels)
num_classes

24

In [46]:
fig , axes = plt.subplots(2,5,figsize=(12,6))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('sample_images.png')
    plt.close()
    print("Sample images saved as 'sample_images.png'.")

Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.
Sample images saved as 'sample_images.png'.


In [47]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

In [48]:
model.compile(
    optimizer ='adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']

)

In [49]:
history = model.fit(
    X_train,y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1
)

Epoch 1/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.2401 - loss: 2.4827 - val_accuracy: 0.6599 - val_loss: 1.2780
Epoch 2/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.5817 - loss: 1.2642 - val_accuracy: 0.8416 - val_loss: 0.5599
Epoch 3/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.7293 - loss: 0.7836 - val_accuracy: 0.9301 - val_loss: 0.2668
Epoch 4/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.8082 - loss: 0.5472 - val_accuracy: 0.9607 - val_loss: 0.1693
Epoch 5/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.8464 - loss: 0.4332 - val_accuracy: 0.9756 - val_loss: 0.1009
Epoch 6/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.8838 - loss: 0.3276 - val_accuracy: 0.9898 - val_loss: 0.0596
Epoch 7/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9060 - loss: 0.2646 - val_accuracy: 0.9774 - val_loss: 0.0684
Epoch 8/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9164 - loss: 0.2350 - val_accu